# Tutorial 1: Quick Start

This notebook mirrors the quick-start flow from the getting started guide.



## What is `floodsr`?

`floodsr` is a flood-depth resolution enhancement tool.
It takes a low-resolution depth raster and reconstructs a higher-resolution result using terrain context from a DEM.

# Using this tutorial

There are three common ways to run this tutorial (i.e., *execution contexts*):
- **command line (CLI)**: proceed by copy/pasting the below commands from your web-browser into your terminal. NOTE for most terminals you'll need to remove the prefix `!` from each command.
- **local notebook (Jupyter)**: Use the <i class="fas fa-download"></i> button to save this notebook as an `.ipynb` file to your local machine (may need to click *save as*) and open it with your local Jupyter kernel. see {ref}`_basic_install_local_notebook` for details.
- **hosted notebook (Colab)**: Use the <i class="fas fa-rocket"></i> button to open this tutorial on Google Colab.


# Install `floodsr`

The appropriate installation steps depend on your execution context, which is described in detail in :ref:`_basic_install`.
Here are the basic steps for each:
- **command line (CLI)**: first, ensure pipx is installed and on PATH, then run `pipx install floodsr`. see {ref}`_basic_install_cli` for more details.
- **local notebook (Jupyter)**: same as above. then either follow the additional steps in {ref}`_basic_install_local_notebook` to properly set up your environment, or be lazy and uncomment the below Jupyter cell and run.
- **hosted notebook (Colab)**: uncomment the below colab cell and run. see {ref}`basic_install_google_colab` for details.

In [1]:
# command line (CLI) install. run `pipx install floodsr`

Local notebook (Jupyter) magic install

In [2]:

# %pip install -q floodsr

Hosted notebook (Colab) install

In [3]:
# !python -m pip install -q floodsr

### prove the install

run the below command to confirm the install worked and `floodsr` is on your PATH:

In [4]:
!floodsr --help

[floodsr-shim] repo_root=/workspace
[floodsr-shim] cache_dir=/workspace/_cache/notebook_tmp/tutorial_1
[floodsr-shim] argv= python -m floodsr.cli --log-level INFO --help
usage: floodsr [-h] [-v] [-q] [--log-level {DEBUG,INFO,WARNING,ERROR}]
               {models,tohr,doctor} ...

Run FloodSR model, cache, and runtime utility commands.

positional arguments:
  {models,tohr,doctor}
    models              List manifest models or fetch cached model weights.
    tohr                Run one super-resolution pass for a low-res depth
                        raster.
    doctor              Report runtime dependency and provider diagnostics.

options:
  -h, --help            show this help message and exit
  -v, --verbose         Increase logging verbosity (repeatable).
  -q, --quiet           Decrease logging verbosity (repeatable).
  --log-level {DEBUG,INFO,WARNING,ERROR}
                        Explicit log level override.


## Download test data

Before running commands, download a small example dataset into your current working directory.

In [5]:
from urllib.request import urlretrieve

urlretrieve(
    "https://github.com/cefect/floodsr/releases/download/v0.0.3/hires002_dem.tif",
    "hires002_dem.tif",
)
urlretrieve(
    "https://github.com/cefect/floodsr/releases/download/v0.0.3/lowres032.tif",
    "lowres032.tif",
)


('lowres032.tif', <http.client.HTTPMessage at 0x7d2c7c0e52e0>)

## List available models

Before running inference, inspect the manifest to see which model versions are available.

In [6]:
!floodsr models list

[floodsr-shim] repo_root=/workspace
[floodsr-shim] cache_dir=/workspace/_cache/notebook_tmp/tutorial_1
[floodsr-shim] argv= python -m floodsr.cli --log-level INFO models list


ResUNet_16x_DEM	model_infer.onnx	https://github.com/cefect/floodsr/releases/download/v2026.02.19/model_infer.onnx


## Fetch model weights

Fetch the default model into the local cache.

In [7]:
!floodsr models fetch --no-show-progress ResUNet_16x_DEM


[floodsr-shim] repo_root=/workspace
[floodsr-shim] cache_dir=/workspace/_cache/notebook_tmp/tutorial_1
[floodsr-shim] argv= python -m floodsr.cli --log-level INFO models fetch --no-show-progress ResUNet_16x_DEM --cache-dir /workspace/_cache/notebook_tmp/tutorial_1


/workspace/_cache/notebook_tmp/tutorial_1/ResUNet_16x_DEM/model_infer.onnx


## Run `tohr` with fetched HRDEM data

If your low-resolution raster falls within HRDEM coverage, you can fetch the high-resolution DEM automatically.

In [8]:
!floodsr tohr --in lowres032.tif --fetch-hrdem

[floodsr-shim] repo_root=/workspace
[floodsr-shim] cache_dir=/workspace/_cache/notebook_tmp/tutorial_1
[floodsr-shim] argv= python -m floodsr.cli --log-level INFO tohr --in lowres032.tif --fetch-hrdem --cache-dir /workspace/_cache/notebook_tmp/tutorial_1


INFO:__main__:starting DEM fetch
  source_id=hrdem
  stac_url=https://datacube.services.geo.ca/api
  collection=hrdem-mosaic-1m
  asset_key=dtm
  gdal_available=True
  force_tiling=False
  fetch_window_size=8192
  memory_limit_gib=16.00
  stac_query_limit=200
  use_project_extent_filter=True
  use_cache=True
  cache_dir=/workspace/_cache/notebook_tmp/tutorial_1
  show_progress=True
  project_extent_url=https://maps-cartes.services.geo.ca/server_serveur/rest/services/NRCan/coverage_HRDEM_en/MapServer/4
  depth_lr_fp=
    /workspace/_cache/notebook_tmp/tutorial_1/run/lowres032.tif


INFO:__main__:found 1 HRDEM item(s) intersecting low-res tile bounds after exact intersection filter
INFO:__main__:raw fetch request grid: width=1,024, height=1,024, pixels=1,048,576, non_windowed_peak_estimate=0.01 GiB
INFO:__main__:HRDEM tile cache state: enabled=1, dir=/workspace/_cache/notebook_tmp/tutorial_1/floodsr_hrdem_tile_cache, existing_files=1, request_token=11afd1814d4c09e5c44a8f97
INFO:__main__:wrote fetched HRDEM tile to
    /workspace/_cache/notebook_tmp/tutorial_1/tmp/floodsr_hrdem_output_11afd1814d4c09e5c44a8f97.tif


INFO:__main__:loaded ORT model 'model_infer.onnx' with providers=['CPUExecutionProvider'] and scale=16
INFO:__main__:tohr path selection
  requested_window_method=feather
  dem_float32_bytes=4,194,304
  windowed_io_threshold_bytes=33,554,432
  selected_platform_materialization=simple


INFO:__main__:starting tohr inference with model_version=ResUNet_16x_DEM
model
    /workspace/_cache/notebook_tmp/tutorial_1/ResUNet_16x_DEM/model_infer.onnx
platform_depth_lr
    /workspace/_cache/notebook_tmp/tutorial_1/tmp/floodsr-platform-prep-cuf24a0a/lowres032_platform_depth.tif
platform_dem_hr
    /workspace/_cache/notebook_tmp/tutorial_1/tmp/floodsr-platform-prep-cuf24a0a/floodsr_hrdem_output_11afd1814d4c09e5c44a8f97_platform_dem.tif
output
    /workspace/_cache/notebook_tmp/tutorial_1/run/lowres032_sr.tif
INFO:__main__:platform-preprocessed inputs
  depth_lr shape=(32, 32) res=(32.0, 32.0) m/pix
  dem_hr shape=(1024, 1024) res=(1.0, 1.0) m/pix
INFO:__main__:model preprocessing complete
INFO:__main__:tohr execution path
  window_method=feather
  execution_path=simple
  model_space_shape=(512, 512)
  raw_output_shape=(1024, 1024)


INFO:__main__:prepared inputs summary:
  aligned depth_lr shape=(32, 32) res=(32.0, 32.0) m/pix
  aligned dem_hr shape=(512, 512) res=(2.0, 2.0) m/pix
  max_depth=5.0
  dem_pct_clip=95.0
INFO:__main__:window config
  method=feather
  overlap_lr=8
  overlap_hr=128
  tile_size_lr=32
  tile_size_hr=512
INFO:__main__:running feather tiling over 1x1 grid
  stride_hr=384
  overlap_hr=128
feather pass:   0%|                                   | 0/1 [00:00<?, ?window/s]

feather pass: 100%|███████████████████████████| 1/1 [00:00<00:00, 19.11window/s]
INFO:__main__:post-resampling model output from (512, 512) to (1024, 1024) on raw DEM grid with bilinear interpolation.
final write pass:   0%|                               | 0/16 [00:00<?, ?block/s]

final write pass: 100%|█████████████████████| 16/16 [00:00<00:00, 350.55block/s]
INFO:__main__:finished tohr inference in 0.157s; wrote 4,624,329 bytes to
    /workspace/_cache/notebook_tmp/tutorial_1/run/lowres032_sr.tif
/workspace/_cache/notebook_tmp/tutorial_1/run/lowres032_sr.tif


## Run `tohr` with a local DEM

If you already have a local DEM, point `floodsr` at it directly.

In [9]:
!floodsr tohr --in lowres032.tif --dem hires002_dem.tif

[floodsr-shim] repo_root=/workspace
[floodsr-shim] cache_dir=/workspace/_cache/notebook_tmp/tutorial_1
[floodsr-shim] argv= python -m floodsr.cli --log-level INFO tohr --in lowres032.tif --dem hires002_dem.tif --cache-dir /workspace/_cache/notebook_tmp/tutorial_1


INFO:__main__:loaded ORT model 'model_infer.onnx' with providers=['CPUExecutionProvider'] and scale=16


INFO:__main__:tohr path selection
  requested_window_method=feather
  dem_float32_bytes=1,048,576
  windowed_io_threshold_bytes=33,554,432
  selected_platform_materialization=simple


INFO:__main__:starting tohr inference with model_version=ResUNet_16x_DEM
model
    /workspace/_cache/notebook_tmp/tutorial_1/ResUNet_16x_DEM/model_infer.onnx
platform_depth_lr
    /workspace/_cache/notebook_tmp/tutorial_1/tmp/floodsr-platform-prep-ddv0fh00/lowres032_platform_depth.tif
platform_dem_hr
    /workspace/_cache/notebook_tmp/tutorial_1/tmp/floodsr-platform-prep-ddv0fh00/hires002_dem_platform_dem.tif
output
    /workspace/_cache/notebook_tmp/tutorial_1/run/lowres032_sr.tif
INFO:__main__:platform-preprocessed inputs
  depth_lr shape=(32, 32) res=(32.0, 32.0) m/pix
  dem_hr shape=(512, 512) res=(2.0, 2.0) m/pix
INFO:__main__:model preprocessing complete
INFO:__main__:tohr execution path
  window_method=feather
  execution_path=simple
  model_space_shape=(512, 512)
  raw_output_shape=(512, 512)
INFO:__main__:prepared inputs summary:
  aligned depth_lr shape=(32, 32) res=(32.0, 32.0) m/pix
  aligned dem_hr shape=(512, 512) res=(2.0, 2.0) m/pix
  max_depth=5.0
  dem_pct_clip=95.0
I

final write pass: 100%|██████████████████| 128/128 [00:00<00:00, 4898.68block/s]
INFO:__main__:finished tohr inference in 0.136s; wrote 1,175,984 bytes to
    /workspace/_cache/notebook_tmp/tutorial_1/run/lowres032_sr.tif
/workspace/_cache/notebook_tmp/tutorial_1/run/lowres032_sr.tif


## Next steps

For more detail, continue with {doc}`../user_guide` or inspect the full {doc}`../cli_reference`.